In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_mlp_regression.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_regression_data_loaders(batch_size=128, test_size=0.3):
    """准备Boston房价回归数据集"""
    logging.info("正在准备Boston房价回归数据集...")
    
    # 加载Boston房价数据集
    df = pd.read_csv('/kaggle/input/the-boston-houseprice-data/boston.csv')
    
    # 检查并处理数据
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    
    # 特征和目标变量
    X = df.drop(['MEDV'], axis=1)  # 特征矩阵
    y = df['MEDV']  # 目标变量（房价中位数）
    
    # 数据标准化
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # 对目标变量标准化
    y = (y - y.mean()) / y.std()
    
    # 划分训练测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values)  
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test.values)   
   
    # 创建DataLoader
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    
    trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    testloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    logging.info("Boston房价回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 测试样本数: {len(test_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return trainloader, testloader, X_train.shape[1]

class MLPRegression(nn.Module):
    """MLP回归模型"""
    def __init__(self, input_dim, hidden_dims=[64, 32], output_dim=1, dropout_rate=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        # 构建隐藏层
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        
        # 输出层
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x).squeeze()

def train_model(model, trainloader, testloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), 
                         lr=hparams['learning_rate'], 
                         weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hparams['epochs'])
    
    best_test_loss = float('inf')
    best_model_state = None
    
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "training_loss_per_epoch": [],
        "test_loss_per_epoch": []
    }

    logging.info("="*30)
    logging.info("  启动 MLP 训练流程")
    logging.info("="*30)
    
    for epoch in range(hparams['epochs']):
        # 训练阶段
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        train_avg_loss = running_loss / len(trainloader)
        log_data["training_loss_per_epoch"].append(train_avg_loss)
        scheduler.step()
        
        # 测试阶段
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        test_avg_loss = test_loss / len(testloader)
        log_data["test_loss_per_epoch"].append(test_avg_loss)
        
        logging.info(f"Epoch {epoch+1}/{hparams['epochs']} | 训练MSE: {train_avg_loss:.4f} | 测试MSE: {test_avg_loss:.4f}")
        
        if test_avg_loss < best_test_loss:
            best_test_loss = test_avg_loss
            best_model_state = model.state_dict()
            logging.info(f"*** 新的最佳测试MSE: {best_test_loss:.4f} ***")
        
        with open(log_filepath, 'w') as f:
            json.dump(log_data, f, indent=4)
        
        if best_test_loss < hparams['loss_target']:
            logging.info(f"目标损失 {hparams['loss_target']} 已达到。")
            break
    
    if best_model_state:
        logging.info(f"训练流程结束。保存最佳模型到 {model_save_path}")
        torch.save({
            'model_state_dict': best_model_state,
            'best_loss': best_test_loss,
            'hyperparameters': hparams
        }, model_save_path)
    else:
        logging.error("没有训练出有效模型，不进行保存。")
    
    return best_test_loss

if __name__ == '__main__':
    # --- 文件与超参数定义 ---
    LOG_FILE = 'training_mlp_regression.log'
    JSON_LOG_FILE = 'training_mlp_regression_log.json'
    MODEL_SAVE_PATH = 'mlp_regression.pth'
    
    HYPERPARAMETERS = {
        "epochs": 200,
        "loss_target": 0.1,
        "learning_rate": 1e-3,
        "weight_decay": 1e-5,
    }

    setup_logging(LOG_FILE)
    logging.info(f"回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, testloader, input_dim = get_regression_data_loaders()
    
    # 创建MLP模型
    mlp_model = MLPRegression(input_dim=input_dim).to(device)

    start_time = time.time()
    best_loss = train_model(
        mlp_model, trainloader, testloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    logging.info(f"最佳测试MSE损失: {best_loss:.4f}")
    
    # 加载并验证保存的最佳模型
    if best_loss < float('inf'):
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳模型")
        logging.info("="*30)
        
        verification_model = MLPRegression(input_dim=input_dim).to(device)
        checkpoint = torch.load(MODEL_SAVE_PATH)
        verification_model.load_state_dict(checkpoint['model_state_dict'])
        verification_model.eval()

        logging.info(f"成功加载模型。保存的MSE损失: {checkpoint['best_loss']:.4f}")

        total_params = sum(p.numel() for p in verification_model.parameters() if p.requires_grad)
        logging.info(f"模型总参数数量: {total_params}")

        test_loss = 0.0
        criterion = nn.MSELoss()
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = verification_model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        avg_test_loss = test_loss / len(testloader)
        logging.info(f"加载后的模型在测试集上的MSE损失: {avg_test_loss:.4f}")
        
        assert abs(avg_test_loss - checkpoint['best_loss']) < 1e-3, "验证失败！"
        logging.info("验证成功！模型可以被正确保存和复用。")
    else:
        logging.info("未能成功训练和保存模型。")

2025-08-14 11:23:52,828 [INFO ]  回归实验超参数: {
    "epochs": 200,
    "loss_target": 0.1,
    "learning_rate": 0.001,
    "weight_decay": 1e-05
}
2025-08-14 11:23:52,829 [INFO ]  正在准备Boston房价回归数据集...
2025-08-14 11:23:52,840 [INFO ]  Boston房价回归数据集准备完毕。
2025-08-14 11:23:52,841 [INFO ]  训练样本数: 354, 测试样本数: 152
2025-08-14 11:23:52,842 [INFO ]  特征维度: 13, 目标维度: 1
2025-08-14 11:23:52,846 [INFO ]  ==============================
2025-08-14 11:23:52,847 [INFO ]    启动 MLP 训练流程
2025-08-14 11:23:52,848 [INFO ]  ==============================
2025-08-14 11:23:52,872 [INFO ]  Epoch 1/200 | 训练MSE: 1.0390 | 测试MSE: 0.8840
2025-08-14 11:23:52,874 [INFO ]  *** 新的最佳测试MSE: 0.8840 ***
2025-08-14 11:23:52,897 [INFO ]  Epoch 2/200 | 训练MSE: 0.7882 | 测试MSE: 0.7911
2025-08-14 11:23:52,898 [INFO ]  *** 新的最佳测试MSE: 0.7911 ***
2025-08-14 11:23:52,919 [INFO ]  Epoch 3/200 | 训练MSE: 0.6612 | 测试MSE: 0.6792
2025-08-14 11:23:52,920 [INFO ]  *** 新的最佳测试MSE: 0.6792 ***
2025-08-14 11:23:52,940 [INFO ]  Epoch 4/200 | 训练MSE: 0.5211 